In [1]:
#after obtaining API location of possible distribution in one country we should filter the results in the following steps
import geopandas as gpd
import pandas as pd
import ast
import osmnx as ox
DATA_DIR = "dataset_big"

In [2]:
#FILTER 1
#keep only the points that fall within Nigerian territory and saves the result to a new GeoPackage file.

points_file = f"{DATA_DIR}/lpg_points_maps_nigeria_3857.gpkg"
boundary_file = f"{DATA_DIR}/Country_boundaries.geojson"

points = gpd.read_file(points_file)
boundary = gpd.read_file(boundary_file)

# Ensure both are in the same CRS
if points.crs != boundary.crs:
    points = points.to_crs(boundary.crs)

# Spatial join: keep points that lie strictly inside one of the states
points_nigeria = gpd.sjoin(points, boundary, how='inner', predicate='within')

# Drop duplicates that may arise if a point falls exactly on a shared border
points_nigeria = points_nigeria[~points_nigeria.index.duplicated(keep='first')]

# Keep only the original columns (optional, for cleanliness)
points_nigeria = points_nigeria[points.columns]

print(f"FILTER 1 – Inside Nigeria")
print(f"  Original points: {len(points)}")
print(f"  Points inside Nigeria: {len(points_nigeria)}")
print(f"  Removed: {len(points) - len(points_nigeria)} "
      f"({100*(len(points)-len(points_nigeria))/len(points):.1f}%)\n")

FILTER 1 – Inside Nigeria
  Original points: 3812
  Points inside Nigeria: 3742
  Removed: 70 (1.8%)



In [3]:
#FILTER 2
#Filter out LNG-related places 


points_no_lng = points_nigeria[~points_nigeria['name'].str.contains('LNG', case=False, na=False)]

print(f"FILTER 2 – Exclude 'LNG' in name")
print(f"  Before: {len(points_nigeria)}")
print(f"  After: {len(points_no_lng)}")
print(f"  Removed: {len(points_nigeria)-len(points_no_lng)} ({100*(len(points_nigeria)-len(points_no_lng))/len(points_nigeria):.1f}%)\n")


FILTER 2 – Exclude 'LNG' in name
  Before: 3742
  After: 3736
  Removed: 6 (0.2%)



In [4]:
#FILTER 3
#Filter removing likely false positives, discard point with suspicius category with also NO key words in the name and NO gas_station as complementary type
            

suspect = [
    'store', 'car_repair', 'food', 'home_goods_store', 'furniture_store',
    'health', 'parking', 'restaurant', 'general_contractor',
    'grocery_or_supermarket', 'electronics_store', 'storage', 'finance',
    'pharmacy', 'supermarket', 'bakery', 'shopping_mall'
]

lpg_key_words = [
    "LPG refill station",
    "gas station with LPG",
    "LPG filling point",
    "LPG dealer",
    "gas cylinder exchange",
    "cooking gas",
    "LPG"
]

def parse_types(types_field):
    if isinstance(types_field, list):
        return types_field
    if isinstance(types_field, str):
        try:
            return ast.literal_eval(types_field)
        except (ValueError, SyntaxError):
            return [t.strip().strip("[]'\"") for t in types_field.split(',')]
    return []

def has_suspect(types_list):
    return any(cat in types_list for cat in suspect)

def has_gas_station(types_list):
    return 'gas_station' in types_list

def has_lpg_keyword(name):
    if pd.isna(name):
        return False
    name_lower = name.lower()
    return any(keyword.lower() in name_lower for keyword in lpg_key_words)

# Prepare types list
points_no_lng = points_no_lng.copy()
points_no_lng['types_list'] = points_no_lng['types'].apply(parse_types)

# Keep if not (suspect AND no gas_station AND no LPG keyword)
keep_mask = ~(
    points_no_lng['types_list'].apply(has_suspect) &
    ~points_no_lng['types_list'].apply(has_gas_station) &
    ~points_no_lng['name'].apply(has_lpg_keyword)
)

points_final = points_no_lng[keep_mask].copy()
points_removed_step3 = points_no_lng[~keep_mask].copy()

# Drop temporary column
points_final.drop(columns=['types_list'], inplace=True)
points_removed_step3.drop(columns=['types_list'], inplace=True)

print(f"FILTER 3 – Remove false positives")
print(f"  Before: {len(points_no_lng)}")
print(f"  After: {len(points_final)}")
print(f"  Removed: {len(points_no_lng)-len(points_final)} ({100*(len(points_no_lng)-len(points_final))/len(points_no_lng):.1f}%)\n")



FILTER 3 – Remove false positives
  Before: 3736
  After: 3523
  Removed: 213 (5.7%)



In [5]:
# FILTER 4 – Proximity to roads OR buildings (data-driven threshold)
# Keep points whose combined distance (min to road or building) 
# is within the 99th percentile of that distribution.

import numpy as np
import geopandas as gpd
import pandas as pd

# Load roads and buildings (already in CRS suitable for metric distances)
roads = gpd.read_file(f"{DATA_DIR}/nigeria_roads_merged.gpkg")
buildings = gpd.read_file(f"{DATA_DIR}/buildings_near_lpg_points_150m.gpkg")

# Convert all to EPSG:3857 (metres)
roads_3857 = roads.to_crs('EPSG:3857')
buildings_3857 = buildings.to_crs('EPSG:3857')
points = points_final.to_crs('EPSG:3857').copy()  # from Filter 3

# Compute distance to nearest road
near_road = gpd.sjoin_nearest(
    points[['geometry']], roads_3857[['geometry']],
    how='left', distance_col='dist_road'
)
near_road = near_road[~near_road.index.duplicated(keep='first')]

# Compute distance to nearest building
near_build = gpd.sjoin_nearest(
    points[['geometry']], buildings_3857[['geometry']],
    how='left', distance_col='dist_building'
)
near_build = near_build[~near_build.index.duplicated(keep='first')]

# Combine distances
points['dist_road'] = near_road['dist_road'].values
points['dist_building'] = near_build['dist_building'].values
points['dist_combined'] = np.minimum(points['dist_road'], points['dist_building'])

# Compute threshold: 99th percentile
threshold = np.percentile(points['dist_combined'], 99)
print(f"99th percentile threshold: {threshold:.1f} m")

# Keep points within the threshold
points_filtered = points[points['dist_combined'] <= threshold].copy()

# Drop temporary distance columns (they are no longer needed)
points_filtered.drop(columns=['dist_road', 'dist_building', 'dist_combined'], inplace=True)

# Statistics
before = len(points)
after = len(points_filtered)
removed = before - after
print("FILTER 4 – Proximity to roads OR buildings (99th percentile)")
print(f"  Points before filter: {before}")
print(f"  Points kept: {after}")
print(f"  Points removed (outliers): {removed} ({100*removed/before:.2f}%)\n")

# Update points_final for the next filter
points_final = points_filtered

99th percentile threshold: 147.5 m
FILTER 4 – Proximity to roads OR buildings (99th percentile)
  Points before filter: 3523
  Points kept: 3487
  Points removed (outliers): 36 (1.02%)



In [6]:
# FILTER 5 – Remove places with no reviews
# Keep only points that have at least one review (user_ratings_total > 0)

before = len(points_final)
points_with_reviews = points_final[points_final['user_ratings_total'] > 0].copy()
after = len(points_with_reviews)
removed = before - after

print("FILTER 5 – Remove places with no reviews")
print(f"  Before: {before}")
print(f"  After: {after}")
print(f"  Removed: {removed} ({100*removed/before:.1f}%)\n")

# Update points_final to the filtered set
points_final = points_with_reviews

FILTER 5 – Remove places with no reviews
  Before: 3487
  After: 2833
  Removed: 654 (18.8%)



In [7]:
#save results

output_final = f"{DATA_DIR}/lpg_points_nigeria_final_filtred.gpkg"
points_final.to_file(output_final, layer='lpg_points', driver='GPKG')
print(f"Final kept points saved to: {output_final}")



Final kept points saved to: dataset_big/lpg_points_nigeria_final_filtred.gpkg
